# 16 famílias restantes: quociente essencial e XOR
Oito famílias já foram excluídas com provas verificadas. Este notebook testa os seis representantes restantes.
Use CPU. Limite de 300 segundos por solver, no máximo dois processos. Não execute junto com outro lote no mesmo ambiente.
Os resultados ficam no runtime até serem baixados. Interromper/desconectar pode perder arquivos. Cada família tem summary.json e logs.
Uma reexecução repete as buscas; não há promessa de retomada automática.


In [ ]:
from pathlib import Path
import subprocess,sys,urllib.request,hashlib,tarfile,json,os
BASE=Path('/content/hrsbf_essential');BASE.mkdir(exist_ok=True)
PROJECT=BASE/'project'
def command(args,cwd=None):
    with subprocess.Popen([str(a) for a in args],cwd=cwd,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1) as p:
        for line in p.stdout: print(line,end='',flush=True)
        code=p.wait()
        if code: raise subprocess.CalledProcessError(code,args)
if not PROJECT.exists(): command(['git','clone','https://github.com/contatofilipevono-rgb/hrsbf-bent-verification.git',PROJECT])
command(['git','fetch','origin','pesquisa-t32-2026-10-04'],PROJECT)
command(['git','checkout','0f7bd4c82cff07aa088f68f920376a26bf1e1f3d'],PROJECT)
solver=BASE/'cryptominisat5'
archive=BASE/'cryptominisat5-v5.16.0-linux-amd64.tar.gz'
if not archive.exists(): urllib.request.urlretrieve('https://github.com/msoos/cryptominisat/releases/download/release/v5.16.0/cryptominisat5-v5.16.0-linux-amd64.tar.gz',archive)
if hashlib.sha256(archive.read_bytes()).hexdigest()!='9599eea560305291cb593ba9d6733605fae763a7f7a1fcdd63191ef8089e9979':raise ValueError('SHA-256 do solver divergente')
with tarfile.open(archive) as tar:
    member=next(m for m in tar.getmembers() if Path(m.name).name=='cryptominisat5' and m.isfile())
    solver.write_bytes(tar.extractfile(member).read())
solver.chmod(0o755)
checker_repo=BASE/'frat-xor'
if not checker_repo.exists(): command(['git','clone','https://github.com/meelgroup/frat-xor.git',checker_repo])
command(['git','checkout','855f3d0ae45fe37c3ad29e4a8ef56e62e1b5e4ad'],checker_repo)
checker=checker_repo/'cake_xlrup/cake_xlrup'
if not checker.exists():command(['make'],checker.parent)
print('Preparado: CPU, solver e checker fixados. Não requer GPU.')


In [ ]:
# Seis representantes cobrem as 16 famílias restantes. 300 s de CPU por solver.
# A preparação, a auditoria e a verificação das provas são adicionais.
from concurrent.futures import ThreadPoolExecutor,as_completed
sys.path.insert(0,str(PROJECT/'pesquisa_t32'))
from modelo_radicais_t32 import directions,free_coordinates
from triagem_familias_t32 import model,coefficient_forms,family_fiber,linear_row
remaining=json.loads((PROJECT/'pesquisa_t32/dezesseis_familias_remanescentes_2026_10_05.json').read_text())
OUT=BASE/'resultados';OUT.mkdir(exist_ok=True)
reps,_,hrows,_=model();forms=None
for group in remaining:
    h=int(group['representative'],16);metadata=OUT/('input_'+format(h,'x')+'.json')
    if metadata.exists():continue
    if forms is None:
        forms={}
        for i,z in enumerate(directions(),1):
            forms[z]=coefficient_forms(reps,z,hrows)
            if i%512==0:print('Formas compartilhadas:',i,flush=True)
    free,sub,_=free_coordinates(h,hrows);fibers=[]
    for z,f in forms.items():
        _,rank,basis=family_fiber(h,hrows,f)
        fibers.append({'z':z,'rank':rank,'radical_basis':basis,'affine_values':[[hex(m),c] for m,c in [sub(linear_row(f,r)) for r in basis]]})
    metadata.write_text(json.dumps({'h':hex(h),'family_dimension':120,'free_original_orbit_indices':free,'fibers':fibers})+'\n')
del forms
summary={'scope':'16 famílias remanescentes; SAT não certifica bentness','completed':False,'families':{}}
def save():
    tmp=OUT/'summary.tmp';tmp.write_text(json.dumps(summary,indent=2)+'\n');tmp.replace(OUT/'summary.json')
def work(group):
    h=group['representative'];dest=OUT/h;dest.mkdir(exist_ok=True)
    args=[sys.executable,str(PROJECT/'pesquisa_t32/certify_essential_fibers.py'),str(OUT/('input_'+h[2:]+'.json')),str(dest),'--solver',str(solver),'--checker',str(checker),'--seconds','300']
    environment=dict(os.environ,CML_HEAP_SIZE='2048',CML_STACK_SIZE='128')
    try:
        with (dest/'runner.log').open('w') as log:
            result=subprocess.run(args,stdout=log,stderr=subprocess.STDOUT,env=environment,timeout=1800)
        report=json.loads((dest/'summary.json').read_text()) if (dest/'summary.json').exists() else {'status':'ERROR','returncode':result.returncode}
    except subprocess.TimeoutExpired:
        report={'status':'RUNNER_TIMEOUT'}
    report['members']=list(group['members']);return h,report
save()
with ThreadPoolExecutor(max_workers=min(2,os.cpu_count() or 1)) as pool:
    for task in as_completed([pool.submit(work,g) for g in remaining]):
        h,report=task.result();summary['families'][h]=report
        summary['new_excluded']=sum(len(r['members']) for r in summary['families'].values() if r['status']=='UNSAT_PROOF_VERIFIED_AND_MODEL_AUDITED')
        save();print(h,report['status'],'novas exclusões:',summary['new_excluded'],flush=True)
summary['completed']=True;save()
print('Lote concluído. Baixe os resultados na próxima célula.')


In [ ]:
import zipfile
from google.colab import files
archive=BASE/'resultado_16_familias.zip'
with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED) as z:
    for p in OUT.rglob('*'):
        if p.is_file():z.write(p,p.relative_to(OUT))
print(json.dumps(json.loads((OUT/'summary.json').read_text()),indent=2))
files.download(str(archive))
